# Q1: Where do fires occur, and how large are they?
One red circle per fire with **positive recorded burned hectares**. **Circle area is directly proportional to hectares**: twice the hectares means twice the circle area. Zero hectares produces no point. Small fires can be extremely small. The linear size legend uses ordinary numbers, such as 100 and 1,000. All circles use one color; transparency reduces overlap. Hover shows municipality, date, incident ID and exact hectares.

Use `CleanData.csv`, currently 1998–2022 for records with hectares. Positions use existing `MUNICIPIOS.csv` municipality reference coordinates because supplied fire coordinates contain values far outside Catalonia. These are approximate municipal locations, not ignition points or fire footprints. Records without municipality reference coordinates are excluded and counted.

Run all cells for the interactive Altair map. The saved image is an offline preview.

In [8]:
import pandas as pd
import altair as alt
import json
from IPython.display import display
alt.data_transformers.disable_max_rows()
fires = pd.read_csv("Clean data/CleanData.csv", parse_dates=["data_incendi"], dtype={"codi_municipi": "string"}).dropna(subset=["total_hectares"])
municipalities = pd.read_csv("dades/MUNICIPIOS.csv", sep=";", encoding="cp1252", dtype="string")
municipalities.index = municipalities.COD_INE.str[:5]
for field, column in [("longitud", "LONGITUD_ETRS89_REGCAN95"), ("latitud", "LATITUD_ETRS89_REGCAN95")]:
    fires[field] = fires.codi_municipi.str.zfill(5).map(pd.to_numeric(municipalities[column].str.replace(",", ".", regex=False)))
print(f"Zero-hectare fires (not plotted): {fires.total_hectares.eq(0).sum()}")
print(f"Excluded unknown locations: {fires.latitud.isna().sum()} of {len(fires):,} fires with hectares.")
fires = fires[fires.total_hectares.gt(0)].dropna(subset=["latitud", "longitud"]).sort_values("total_hectares", ascending=False)
with open("Catalonia.geojson", encoding="utf-8") as f: catalonia = json.load(f)

Zero-hectare fires (not plotted): 26
Excluded unknown locations: 15 of 15,366 fires with hectares.


In [9]:
outline = alt.Chart(alt.Data(values=catalonia["features"])).mark_geoshape(fill="#f4f3ee", stroke="gray")
dots = alt.Chart(fires).mark_circle(color="#b91c1c", opacity=0.45, strokeWidth=0).encode(
    longitude="longitud:Q", latitude="latitud:Q", size=alt.Size("total_hectares:Q", title="Burned area (ha)", scale=alt.Scale(type="linear", domain=[0, fires.total_hectares.max()], range=[0, 2400]), legend=alt.Legend(values=[100, 1000, 10000], format=",.0f")),
    tooltip=[alt.Tooltip("municipi:N", title="Municipality"), alt.Tooltip("data_incendi:T", title="Date"), alt.Tooltip("total_hectares:Q", title="Burned hectares", format=",.2f"), alt.Tooltip("id_incendi:N", title="Incident")]
)
chart = (outline + dots).project(type="mercator").properties(width=650, height=500, title="Fires with recorded burned area in Catalonia (1998–2022)")
spec = chart.to_dict()
spec["$schema"] = "https://vega.github.io/schema/vega-lite/v5.json"
display({"application/vnd.vegalite.v5+json": spec}, raw=True)

## Fire frequency · All merged records, 1998–2024
One circle per municipality; **circle area is proportional to the number of fires**, not hectares. This includes both GovEsp and GenCat records from the merged dataset, including zero or missing hectares. Hover shows the municipality and count.

Locations use municipality reference coordinates, not exact ignition points. Unmapped locations are reported and excluded from the map. Aggregation removes coincident incident marks, although neighbouring circles may still overlap. The first map describes recorded burned area; this second map describes fire frequency across the full project period. The reduced circle-size range is retained.


In [10]:
all_fires = pd.read_csv("Clean data/CleanData.csv", parse_dates=["data_incendi"], dtype={"codi_municipi": "string"})
all_fires = all_fires[all_fires.data_incendi.dt.year.between(1998, 2024)].copy()
for field, column in [("longitud", "LONGITUD_ETRS89_REGCAN95"), ("latitud", "LATITUD_ETRS89_REGCAN95")]:
    all_fires[field] = all_fires.codi_municipi.str.zfill(5).map(pd.to_numeric(municipalities[column].str.replace(",", ".", regex=False)))
located = all_fires.dropna(subset=["latitud", "longitud"])
print(f"Mapped {len(located):,} of {len(all_fires):,} fires; {len(all_fires) - len(located)} have unknown municipality locations.")
frequency = located.groupby("codi_municipi").agg(municipi=("municipi", "first"), longitud=("longitud", "first"), latitud=("latitud", "first"), count=("id_incendi", "size")).reset_index().sort_values("count", ascending=False)


Mapped 16,456 of 16,471 fires; 15 have unknown municipality locations.


In [12]:
frequency_dots = alt.Chart(frequency).mark_circle(color="#b91c1c", opacity=0.5, strokeWidth=0).encode(
    longitude="longitud:Q", latitude="latitud:Q",
    size=alt.Size("count:Q", title="Number of fires", scale=alt.Scale(type="linear", domain=[0, frequency["count"].max()], range=[0, 800]), legend=alt.Legend(format=",.0f")),
    tooltip=[alt.Tooltip("municipi:N", title="Municipality"), alt.Tooltip("count:Q", title="Fires", format=",.0f")]
)
frequency_map = (outline + frequency_dots).project(type="mercator").properties(width=650, height=500, title=alt.TitleParams(text="Q1 · Fire frequency by municipality (1998–2024)", subtitle="GovEsp + GenCat · All fires with known municipality locations"))
spec = frequency_map.to_dict()
spec["$schema"] = "https://vega.github.io/schema/vega-lite/v5.json"
display({"application/vnd.vegalite.v5+json": spec, "text/html": frequency_map.to_html()}, raw=True)


<!DOCTYPE html>